In [ ]:
import pyperclip
import re
from ipywidgets import Button, Output, VBox, IntSlider, HBox, Label
from IPython.display import display, clear_output
# Добавляем импорт для эмуляции клавиш
import keyboard
import time
from threading import Thread

# === 1. Загрузка файла ===
file_path = r"C:\Users\lutsevich\Desktop\py\wt_stats\wt_stats_parser\report_dump.txt"

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        full_text = f.read()
    print(f"✅ Файл загружен: {file_path}")
except FileNotFoundError:
    print(f"❌ Файл не найден: {file_path}")
    full_text = ""

In [ ]:
# === 2. Извлечение блоков ===
normalized_text = full_text.replace('\r\n', '\n').replace('\r', '\n')
lines = normalized_text.split('\n')

blocks = []  # Список кортежей (timestamp, content)
i = 0

while i < len(lines):
    if lines[i].startswith("=================================================="):
        if i + 1 < len(lines) and re.match(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', lines[i+1].strip()):
            if i + 2 < len(lines) and lines[i+2].startswith("=================================================="):
                timestamp = lines[i+1].strip()
                block_start = i + 3
                
                j = block_start
                while j < len(lines):
                    if lines[j].startswith("--------------------------------------------------"):
                        content_lines = lines[block_start:j]
                        content = '\n'.join(content_lines).strip()
                        if content:
                            blocks.append((timestamp, content))
                        i = j
                        break
                    j += 1
                else:
                    content_lines = lines[block_start:]
                    content = '\n'.join(content_lines).strip()
                    if content:
                        blocks.append((timestamp, content))
                    i = len(lines)
            else:
                i += 1
        else:
            i += 1
    else:
        i += 1

print(f"📊 Найдено блоков: {len(blocks)}")

In [ ]:
# === 3. Интерфейс Jupyter Notebook ===
if blocks:
    slider = IntSlider(value=1, min=1, max=len(blocks), description='Бой:')
    copy_button = Button(description="Копировать в буфер")
    next_button = Button(description="Следующий →")
    prev_button = Button(description="← Предыдущий")
    info_label = Label(value=f"Бой {1} из {len(blocks)}: {blocks[0][0]}")
    output = Output()
    
    def update_info():
        idx = slider.value - 1
        info_label.value = f"Бой {slider.value} из {len(blocks)}: {blocks[idx][0]}"
    
    def on_slider_change(change):
        update_info()
    
    # Функция для эмуляции Ctrl+C в отдельном потоке
    def simulate_ctrl_c():
        time.sleep(0.5)  # Небольшая задержка, чтобы GUI обновился
        keyboard.press_and_release('ctrl+c')
        # print("Симуляция Ctrl+C выполнена") # Для отладки
    
    def on_copy_clicked(b):
        with output:
            clear_output()
            idx = slider.value - 1
            text_to_copy = blocks[idx][1]
            pyperclip.copy(text_to_copy)
            print(f"✅ Блок #{slider.value} скопирован в буфер обмена")
            print(f"⏰ Время: {blocks[idx][0]}")
            print(f"📏 Длина: {len(text_to_copy)} символов")
            
            # 1. Нажимаем "Следующий" если это не последний блок
            if slider.value < len(blocks):
                slider.value += 1
            
            # 2. Эмулируем Ctrl+C в отдельном потоке
            # Это предотвращает "зависание" GUI
            thread = Thread(target=simulate_ctrl_c)
            thread.start()
    
    def on_next_clicked(b):
        if slider.value < len(blocks):
            slider.value += 1
    
    def on_prev_clicked(b):
        if slider.value > 1:
            slider.value -= 1
    
    slider.observe(on_slider_change, names='value')
    copy_button.on_click(on_copy_clicked)
    next_button.on_click(on_next_clicked)
    prev_button.on_click(on_prev_clicked)
    
    controls = HBox([prev_button, slider, next_button])
    buttons = HBox([copy_button])
    
    display(VBox([controls, info_label, buttons, output]))
    update_info()
    
else:
    print("❌ Не удалось найти блоки боёв в файле")